# 5. Segment and quantify

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SMLCI/acia-core/blob/main/docs/tutorials/05_segment_and_quantify.ipynb)

This is the one that pays for the previous four. We take the raw time-lapse,
segment every cell with a deep-learning model, measure them in physical units,
throw out the artefacts, and end with a **growth rate** — the kind of number that
goes in a figure.

Notably, none of this needs tracking. {func}`~acia.analysis.extract_growth`
aggregates cell area per timepoint, so you get a population growth rate from
segmentation alone. Tracking is what you add when you want *per-lineage*
answers.

:::{tip}
No GPU required. Omnipose's `bact_phase_omni` is a small model — the whole
notebook runs in about a minute on a plain CPU, so it works on a laptop or a free
Colab runtime. A GPU makes it faster, not possible.
:::

In [ ]:
# On Colab (or any fresh environment) this installs acia.
# Locally, if you already have acia installed, it is a no-op.
try:
    import acia  # noqa: F401
except ImportError:
    %pip install -q acia

## Install a segmentation backend

We use **Omnipose** with its `bact_phase_omni` model, which is trained on exactly
this kind of imagery: bacteria in phase contrast. It is also small (a 25 MB
model) and fast enough on a CPU that this notebook runs without a GPU in about a
minute.

`acia` supports several backends, but they pin conflicting versions of
`cellpose`, `torch` and `numpy`, so **exactly one can be installed per
environment** — see {doc}`/installation`. Swapping backends is a one-line change
in the cell further down; swapping environments is the price.

In [ ]:
try:
    import omnipose  # noqa: F401
except ImportError:
    %pip install -q omnipose==1.0.6 natsort

In [ ]:
from pathlib import Path
from urllib.request import (
    HTTPBasicAuthHandler,
    HTTPPasswordMgrWithDefaultRealm,
    build_opener,
    install_opener,
    urlretrieve,
)

# A public ownCloud share. The share token acts as the username, so single
# frames can be fetched over WebDAV instead of downloading the whole 800-frame,
# ~800 MB archive.
SHARE = "D0A9ftcpqwKm1Rq"
BASE = "https://fz-juelich.sciebo.de/public.php/webdav"
SEQUENCE = Path("data/colony")

N_FRAMES = 20  # of 800 available
FRAME_STEP = 20  # every 20th frame -> 20 minutes between the frames we keep

mgr = HTTPPasswordMgrWithDefaultRealm()
mgr.add_password(None, BASE, SHARE, "")
install_opener(build_opener(HTTPBasicAuthHandler(mgr)))

SEQUENCE.mkdir(parents=True, exist_ok=True)
for i in range(N_FRAMES):
    name = f"t{i * FRAME_STEP:04d}.tif"
    target = SEQUENCE / name
    if not target.exists():
        urlretrieve(f"{BASE}/00/{name}", target)

print(SEQUENCE, "->", len(list(SEQUENCE.glob("*.tif"))), "frames (~1 MB each)")

## Load the sequence

We work with all 20 downloaded frames at full resolution — Omnipose is cheap
enough that there is no need to shrink further here. On a longer sequence, or a
heavier backend, the subsampling habit from
[tutorial 2](02_the_sequence_model.ipynb) is what keeps iteration fast.

In [ ]:
import torch

print("accelerator:", "cuda" if torch.cuda.is_available() else "none (CPU)")
print("Omnipose runs fine either way -- roughly 3 s per frame on a CPU.")

In [ ]:
from acia import ureg
from acia.segm.open import open_sequence

src = open_sequence(SEQUENCE).position(0)
src = src.with_pixel_size(0.072 * ureg.micrometer).with_frame_interval(20 * ureg.minute)

print(f"{len(src)} frames, {src.size_h}x{src.size_w} px")
print("time span:", src.timepoints[-1].to("hour"))

## Segment

A segmenter is a callable: hand it a source, get back an
{class}`~acia.base.Overlay` of detections. The model is loaded lazily on first
use and, by default, released afterwards so the GPU memory goes back to the
system.

In [ ]:
from acia.segm.processor.omnipose import OmniposeSegmenter

segmenter = OmniposeSegmenter(model="bact_phase_omni")

overlay = segmenter(src)

print(len(overlay), "detections across", overlay.numFrames(), "frames")

Each detection knows its frame, its id and its area in **pixels** — the raw
geometric measurement, before any calibration is applied.

One thing a segmenter does *not* do is attach a time model: `contour.time` is
`None` until the overlay is told what the frames mean. (Trackers do this for you;
a bare segmentation does not.) Attaching it is one call, and worth doing because
it makes every detection self-describing.

In [ ]:
first = next(iter(overlay))
print(
    "before:", first.frame, first.id, round(first.area, 1), "px^2, time =", first.time
)

overlay = overlay.with_timepoints(src.timepoints)

first = next(iter(overlay))
print(
    "after :", first.frame, first.id, round(first.area, 1), "px^2, time =", first.time
)

## See what the model did

Never trust a segmentation you have not looked at.
{func}`~acia.viz.render_segmentation_mask` paints the masks over the image and,
as always, returns a source — so it composes with the annotation and video
helpers from [tutorial 3](03_look_at_your_data.ipynb).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from acia.viz import render_segmentation_mask

painted = render_segmentation_mask(src.to_rgb(), overlay, alpha=0.5)

fig, axes = plt.subplots(1, 3, figsize=(12, 4.2))
for ax, idx in zip(axes, [0, len(src) // 2, len(src) - 1], strict=True):
    ax.imshow(np.asarray(painted[idx].raw))
    ax.set_title(f"{src.timepoints[idx].to('hour'):~.1f}")
    ax.axis("off")
fig.suptitle("Omnipose segmentation")
fig.tight_layout()
plt.show()

In [ ]:
from IPython.display import Video

from acia.viz import render_video

render_video(painted, "segmented.mp4", framerate=4)
Video("segmented.mp4", embed=True, width=420)

## Measure

{class}`~acia.analysis.ExtractorExecutor` turns the overlay into a tidy,
id-indexed DataFrame — one row per detection, one column per property. Because
the source is calibrated, areas come out in µm² and times in hours without any
further configuration.

In [ ]:
from acia.analysis import (
    AreaEx,
    BoundaryClosenessEx,
    CircularityEx,
    ExtractorExecutor,
    FrameEx,
    PerimeterEx,
    TimeEx,
)

properties = ExtractorExecutor().execute(
    overlay,
    src,
    extractors=[
        FrameEx(),
        TimeEx(),
        AreaEx(),
        PerimeterEx(),  # CircularityEx is derived from area and perimeter,
        CircularityEx(),  # so PerimeterEx must come before it
        BoundaryClosenessEx(),
    ],
)

print(properties.head())
print()
print("units:", properties.attrs["units"])

## Throw out the artefacts

Real segmentations contain debris and merged blobs.
{func}`~acia.segm.filter.apply_cell_filters` takes the measured table and a list
of filters, with thresholds in physical units — which is the whole reason we
bothered with calibration.

In [ ]:
from acia.segm.filter import AreaFilter, BoundaryClosenessFilter, apply_cell_filters

filters = [
    # a C. glutamicum cell is roughly 1-3 um^2; anything far below that is debris
    AreaFilter(vmin=0.3 * ureg.micrometer**2, vmax=10 * ureg.micrometer**2),
    # a cell clipped by the edge of the image has a meaningless area
    BoundaryClosenessFilter(min_distance=1 * ureg.micrometer),
]

filtered_overlay = apply_cell_filters(overlay, filters, properties=properties)

print(
    f"{len(overlay)} detections -> {len(filtered_overlay)} kept "
    f"({len(overlay) - len(filtered_overlay)} removed)"
)

{func}`~acia.analysis.properties.plot_property_histograms` shows the before and
after distributions together, so you can see what a threshold actually did rather
than guessing.

In [ ]:
from acia.analysis.properties import plot_property_histograms

properties_after = ExtractorExecutor().execute(
    filtered_overlay,
    src,
    extractors=[FrameEx(), TimeEx(), AreaEx(), PerimeterEx(), CircularityEx()],
)

plot_property_histograms(
    properties,
    ["area", "circularity"],
    df_after=properties_after,
    show_removed=True,
)
plt.show()

## The growth rate

{func}`~acia.analysis.extract_growth` does the last step in one call: it
aggregates total cell area per timepoint, fits an exponential model, and returns
the table, the fit result and a figure.

In [ ]:
from acia.analysis import extract_growth

table, result, figure = extract_growth(filtered_overlay, src, time_unit="hour")

print(table.head())
print()
print("growth rate  :", result.growth_rate)
print("doubling time:", result.doubling_time)
print("R^2          :", round(result.r_squared, 4))
plt.show()

## Filters are only safe when they are uncorrelated with the answer

Both filters above improved the fit rather than distorting it — the R² went from
0.991 unfiltered to 1.000. That is the outcome you want, and it is not automatic.

`BoundaryClosenessFilter` is a good example of a filter whose correctness depends
entirely on the data. Here the colony grows in the middle of a cultivation
chamber, so whether a cell touches the image border is essentially independent of
how much the colony has grown — dropping those cells removes noise and nothing
else.

Point the same filter at a dense field where cells cover the whole frame and it
does the opposite: border-touching becomes *correlated* with growth, so the
filter removes the signal along with the artefacts, and you still get a
confident-looking growth rate that is simply wrong.

The rule worth carrying: a filter is safe only when what it removes is
**independent of the quantity you are measuring**. Always compare the
before/after distributions *and* the resulting fit, rather than applying a filter
because it sounds prudent.

:::{note}
Treat the exact number with appropriate caution: this is one microcolony, 20
frames out of 800, segmented with a stock model and no parameter tuning. That
said, ~1.3 h is a plausible doubling time for *C. glutamicum* under these
conditions, and the fit is tight. What matters for the tutorial is that the
*pipeline* is complete and every quantity carries its unit — scaling up to the
full sequence, a tuned model, or hundreds of positions is a matter of changing
parameters, not code.
:::

## Where to go next

You now have the full loop: **open → slice → visualize → segment → measure →
filter → quantify.**

* **Tracking and lineages.** Add a tracker
  ({class}`~acia.tracking.processor.trackastra.TrackastraTracker`,
  `UltrackTracker`, `LaptrackTracker`, `PyUATTracker`) to follow individual cells
  through divisions, then plot lineage trees and per-cell doubling times.
* **A different backend.** Replace `CellposeSAMSegmenter` with
  `CellposeSAMSegmenter` (a strong generalist, but far heavier on CPU),
  `CellposeSegmenter`, `CPNSegmenter` or `YOLOSegmenter` — same call signature, different environment.
* **Scale it up.** {func}`acia.analysis.scale` runs this notebook once per
  sequence across hundreds of positions; see {doc}`/guide/scaling`.
* **Real experiments.** The
  [acia-workflows](https://github.com/JuBiotech/acia-workflows)
  collection has complete published analyses — growth-rate quantification,
  fluorescence co-culture, single-cell oxygen response — built on exactly these
  pieces.